# HyperParameter Tunning (CBC)

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import optuna
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone

In [ ]:
df = pd.read_csv(r"train.csv")

df

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699630,699630,57,612,2,1,1,1,1,3,3,...,2,2,4,0,0.0,Female,Loyal Customer,Business travel,Eco,False
699631,699631,30,862,2,2,2,3,4,2,4,...,4,2,4,24,34.0,Female,disloyal Customer,Business travel,Eco,False
699632,699632,49,3148,5,5,5,5,4,4,4,...,5,5,5,0,0.0,Male,Loyal Customer,Business travel,Business,True
699633,699633,61,679,5,2,2,2,5,4,4,...,5,5,1,0,0.0,Female,Loyal Customer,Business travel,Eco,True


In [ ]:
def get_X_y(df) :

    ranges = [0 , 10 , 20 , 30 , 40 , 50 , 60 , 70 , 80 , 90 , 100 , 200 , 300 , 400 , 500]

    j = 1

    for i in range(len(ranges) - 1):

        condition1 = (
        (df["Departure Delay in Minutes"] > ranges[i]) &
        (df["Departure Delay in Minutes"] <= ranges[j]) &
        (df["Arrival Delay in Minutes"].isnull())
        )

        condition2 = (
        (df["Departure Delay in Minutes"] > ranges[i]) &
        (df["Departure Delay in Minutes"] <= ranges[j])
        )

        col = "Arrival Delay in Minutes"

        fill = float((df["Departure Delay in Minutes"][condition2]).mean())

        fill = round(fill , 2)

        df.loc[condition1 , col] = fill

        j = j + 1

    df.loc[df["Arrival Delay in Minutes"].isnull() , "Arrival Delay in Minutes"] = 0

    df["Total Delay"] = df["Departure Delay in Minutes"] + df["Arrival Delay in Minutes"]

    df = df.drop(["Arrival Delay in Minutes" , "Departure Delay in Minutes" ] , axis = 1)

    X = df.drop(["satisfaction" , "id"] , axis = 1)
    y = df[["satisfaction"]]

    y = OrdinalEncoder().fit_transform(y)
    y = (pd.Series(y.flatten())).astype(int)

    return X , y

In [ ]:
X_train , y_train = get_X_y(df)

X_train , X_test , y_train , y_test = train_test_split(
    X_train ,
    y_train ,
    test_size = 0.2 ,
    random_state = 42 ,
    stratify = y_train
)

In [ ]:
cat_cols = X_train.select_dtypes(include = ["string" , "object" , "category"]).columns

preprocess = ColumnTransformer([
    ("cat" , OneHotEncoder(handle_unknown = "ignore") , cat_cols)
] , remainder = "passthrough")

model = Pipeline([
    ("preprocess" , preprocess) ,
    ("cbc" , CatBoostClassifier())
])

In [ ]:
# 1st Try

def objective(trial):

    param = {

        "cbc__iterations": trial.suggest_int(
            "iterations",
            800,
            4000
        ),

        "cbc__learning_rate": trial.suggest_float(
            "learning_rate",
            0.005,
            0.20,
            log=True
        ),

        "cbc__depth": trial.suggest_int(
            "depth",
            4,
            12
        ),

        "cbc__l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            0.01,
            100,
            log=True
        ),

        "cbc__random_strength": trial.suggest_float(
            "random_strength",
            0,
            10
        ),

        "cbc__border_count": trial.suggest_int(
            "border_count",
            64,
            256
        ),

        "cbc__boosting_type": trial.suggest_categorical(
            "boosting_type",
            ["Plain"]
        ),

        "cbc__grow_policy": trial.suggest_categorical(
            "grow_policy",
            ["SymmetricTree", "Depthwise", "Lossguide"]
        ),

        "cbc__eval_metric": "AUC",
        "cbc__loss_function": "Logloss",
        "cbc__verbose": 0,
        "cbc__random_state": 42,
        "cbc__task_type": "GPU",
        "cbc__devices": "0"
    }

    trial_model = Pipeline([
        ("preprocess", clone(preprocess)),
        ("cbc", CatBoostClassifier())
    ])

    trial_model.set_params(**param)

    trial_model.fit(
        X_train,
        y_train
    )

    y_proba = trial_model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)

    return auc


study = optuna.create_study(
    study_name="STUDY 1",
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=30
)

print("Best ROC-AUC:", study.best_value)
print("Best Parameters:")
print(study.best_params)

[I 2026-10-13 16:10:23,446] A new study created in memory with name: STUDY 1
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-10-13 16:10:56,459] Trial 0 finished with value: 0.9546246331533838 and parameters: {'iterations': 1997, 'learning_rate': 0.006022888742393706, 'depth': 5, 'l2_leaf_reg': 0.0507895129863037, 'random_strength': 5.587050754414743, 'border_count': 234, 'boosting_type': 'Plain', 'grow_policy': 'SymmetricTree'}. Best is trial 0 with value: 0.9546246331533838.
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-10-13 16:11:27,292] Trial 1 finished with value: 0.9567848186226771 and parameters: {'iterations': 3363, 'learning_rate': 0.015443854204201388, 'depth': 5, 'l2_leaf_reg': 0.207990856874492, 'random_strength': 7.834804279320665, 'border_count': 118, 'boosting_type': 'Plain', 'grow_policy': 'SymmetricTree'}. Best is trial 1 with value: 0.9567848186226771.
Default metric period is 5 because AUC is/are not impl

Best ROC-AUC: 0.9578835125523726
Best Parameters:
{'iterations': 3696, 'learning_rate': 0.005592724757272028, 'depth': 9, 'l2_leaf_reg': 0.04599948484682543, 'random_strength': 1.6691327101631093, 'border_count': 229, 'boosting_type': 'Plain', 'grow_policy': 'Depthwise'}


In [ ]:
# 1st Try

def objective(trial):

    param = {

        "cbc__iterations": trial.suggest_int(
            "iterations",
            800,
            4000
        ),

        "cbc__learning_rate": trial.suggest_float(
            "learning_rate",
            0.005,
            0.20,
            log=True
        ),

        "cbc__depth": trial.suggest_int(
            "depth",
            4,
            12
        ),

        "cbc__l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            0.01,
            100,
            log=True
        ),

        "cbc__random_strength": trial.suggest_float(
            "random_strength",
            0,
            10
        ),

        "cbc__border_count": trial.suggest_int(
            "border_count",
            64,
            256
        ),

        "cbc__boosting_type": trial.suggest_categorical(
            "boosting_type",
            ["Plain"]
        ),

        "cbc__grow_policy": trial.suggest_categorical(
            "grow_policy",
            ["SymmetricTree", "Depthwise", "Lossguide"]
        ),

        "cbc__eval_metric": "AUC",
        "cbc__loss_function": "Logloss",
        "cbc__verbose": 0,
        "cbc__random_state": 42,
        "cbc__task_type": "GPU",
        "cbc__devices": "0"
    }

    trial_model = Pipeline([
        ("preprocess", clone(preprocess)),
        ("cbc", CatBoostClassifier())
    ])

    trial_model.set_params(**param)

    trial_model.fit(
        X_train,
        y_train
    )

    y_proba = trial_model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)

    return auc


study = optuna.create_study(
    study_name="STUDY 1",
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=30
)

print("Best ROC-AUC:", study.best_value)
print("Best Parameters:")
print(study.best_params)

[I 2026-10-13 16:52:49,499] A new study created in memory with name: STUDY 1
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-10-13 16:53:57,740] Trial 0 finished with value: 0.9573281428903001 and parameters: {'iterations': 3542, 'learning_rate': 0.00952740732433604, 'depth': 8, 'l2_leaf_reg': 0.08381520455406613, 'random_strength': 1.5057500633875542, 'border_count': 174, 'boosting_type': 'Plain', 'grow_policy': 'Lossguide'}. Best is trial 0 with value: 0.9573281428903001.
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-10-13 16:54:34,772] Trial 1 finished with value: 0.9560872421221863 and parameters: {'iterations': 2907, 'learning_rate': 0.008337583398272778, 'depth': 4, 'l2_leaf_reg': 1.6598580672081233, 'random_strength': 0.519898819793474, 'border_count': 92, 'boosting_type': 'Plain', 'grow_policy': 'Lossguide'}. Best is trial 0 with value: 0.9573281428903001.
Default metric period is 5 because AUC is/are not implemented

Best ROC-AUC: 0.9581059989132716
Best Parameters:
{'iterations': 1971, 'learning_rate': 0.008513323279531235, 'depth': 10, 'l2_leaf_reg': 12.578821933374982, 'random_strength': 9.049273412474395, 'border_count': 251, 'boosting_type': 'Plain', 'grow_policy': 'Depthwise'}


In [ ]:
# 2nd Try

def objective(trial):

    param = {

        "cbc__iterations": trial.suggest_int(
            "iterations",
            1500,
            4000
        ),

        "cbc__learning_rate": trial.suggest_float(
            "learning_rate",
            0.005,
            0.05,
            log=True
        ),

        "cbc__depth": trial.suggest_int(
            "depth",
            7,
            14
        ),

        "cbc__l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            5,
            30,
            log=True
        ),

        "cbc__random_strength": trial.suggest_float(
            "random_strength",
            0,
            10
        ),

        "cbc__border_count": trial.suggest_int(
            "border_count",
            64,
            256
        ),


        "cbc__boosting_type" : "Plain" ,
        "cbc__grow_policy" : "Depthwise" ,
        "cbc__eval_metric": "AUC",
        "cbc__loss_function": "Logloss",
        "cbc__verbose": 0,
        "cbc__random_state": 42,
        "cbc__task_type": "GPU",
        "cbc__devices": "0"
    }

    trial_model = Pipeline([
        ("preprocess", clone(preprocess)),
        ("cbc", CatBoostClassifier())
    ])

    trial_model.set_params(**param)

    trial_model.fit(
        X_train,
        y_train
    )

    y_proba = trial_model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)

    return auc


study = optuna.create_study(
    study_name="STUDY 1",
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=30
)

print("Best ROC-AUC:", study.best_value)
print("Best Parameters:")
print(study.best_params)

[I 2026-10-14 15:20:19,536] A new study created in memory with name: STUDY 1
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-10-14 15:28:16,641] Trial 0 finished with value: 0.9561985704543249 and parameters: {'iterations': 3695, 'learning_rate': 0.007628164905121778, 'depth': 12, 'l2_leaf_reg': 5.131523680686453, 'random_strength': 0.03520446754793016, 'border_count': 211}. Best is trial 0 with value: 0.9561985704543249.
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-10-14 15:30:41,844] Trial 1 finished with value: 0.9545393977612177 and parameters: {'iterations': 2740, 'learning_rate': 0.04719788689407403, 'depth': 10, 'l2_leaf_reg': 17.77850149325906, 'random_strength': 7.226362644156632, 'border_count': 233}. Best is trial 0 with value: 0.9561985704543249.
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2026-10-14 15:39:17,800] Trial 2 finished with value: 0.9554806252145828 and parameters: {'iter

Best ROC-AUC: 0.9580975836915557
Best Parameters:
{'iterations': 2228, 'learning_rate': 0.007136331918590656, 'depth': 10, 'l2_leaf_reg': 27.504619697680592, 'random_strength': 8.092912191224945, 'border_count': 237}
